# L4S_05 — Comparación de Modelos · 5-Fold CV · Dataset Completo (3 799 parches)

**Proyecto:** Detección de Deslizamientos — Landslide4Sense  
**Objetivo:** Comparar métricas finales de todos los modelos entrenados bajo el mismo protocolo 5-Fold.

| Modelo | Tipo | Nivel evaluación |
|--------|------|-----------------|
| Logistic Regression | Clásico | Patch |
| SVM (RBF) | Clásico | Patch |
| Random Forest | Clásico | Patch |
| ResNet-50 | Deep Learning | Patch |
| EfficientNet-B4 | Deep Learning | Patch |
| U-Net ResNet-34 | Deep Learning | Píxel |

> **Nota:** Los clásicos y los modelos de clasificación DL operan a nivel de parche (positivo/negativo).  
> La U-Net opera a nivel de píxel (segmentación). Las métricas no son directamente comparables,  
> pero se presentan juntas para una visión global del proyecto.


In [ ]:
# ── Celda 0: Entorno y Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

DRIVE_PATH = '/content/drive/MyDrive/Landslide4Sense'
ROOT = Path(DRIVE_PATH)
OUT_DIR = ROOT / 'results' / 'comparable_literatura' / 'comparacion_modelos'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'✅ Salida: {OUT_DIR}')


## 1. Carga de resultados L4S desde Drive

In [ ]:
# ── Celda 1: Carga de JSONs por modelo ──────────────────────────────────────
def load_json(rel_path):
    p = ROOT / rel_path
    if p.exists():
        with open(p) as f: return json.load(f)
    print(f'⚠️  No encontrado: {p}'); return None

# Clásicos (L4S_01)
_cls = load_json('results/comparable_literature/classicos_5fold/classicos_5fold_results.json')

# ResNet-50 (L4S_02)
_rn  = load_json('results/comparable_literature/resnet50_5fold/kfold5_summary.json')

# EfficientNet-B4 (L4S_03)
_eff = load_json('results/comparable_literatura/efficientnet_5fold/kfold5_summary.json')

# U-Net (L4S_04) — un JSON por fold
_unet_folds = []
for k in range(1, 6):
    d = load_json(f'results/comparable_literatura/unet_5fold/fold{k}_results.json')
    if d: _unet_folds.append(d)

# ── Construir tabla unificada ────────────────────────────────────────────────
MODELOS = []

COLORES = {
    'Logistic Regression': '#6EE7B7',
    'SVM (RBF)':           '#34D399',
    'Random Forest':       '#059669',
    'ResNet-50':           '#818CF8',
    'EfficientNet-B4':     '#A78BFA',
    'U-Net ResNet-34':     '#F59E0B',
}

if _cls:
    for key, nombre in [('logistic_regression','Logistic Regression'),
                         ('svm_rbf','SVM (RBF)'),
                         ('random_forest','Random Forest')]:
        d = _cls['models'].get(key, {})
        folds_f1 = [f['f1'] for f in d.get('folds', [])]
        MODELOS.append({
            'nombre': nombre, 'tipo': 'Clásico', 'nivel': 'patch',
            'f1_mean': d.get('mean_f1', np.nan),
            'f1_std':  d.get('std_f1',  np.nan),
            'auc_mean': d.get('mean_auc_roc', np.nan),
            'aupr_mean': d.get('mean_auc_pr', np.nan),
            'iou_mean': d.get('mean_iou', np.nan),
            'folds_f1': folds_f1,
        })

if _rn:
    ag = _rn['aggregate']
    folds_f1 = [f['f1_thr05'] for f in _rn.get('folds', [])]
    MODELOS.append({
        'nombre': 'ResNet-50', 'tipo': 'Deep Learning', 'nivel': 'patch',
        'f1_mean': ag.get('mean_f1_thr05', np.nan),
        'f1_std':  ag.get('std_f1_thr05',  np.nan),
        'auc_mean': ag.get('mean_auc_roc', np.nan),
        'aupr_mean': ag.get('mean_auc_pr', np.nan),
        'iou_mean': np.nan,
        'folds_f1': folds_f1,
    })

if _eff:
    ag = _eff['aggregate']
    folds_f1 = [f['f1_thr05'] for f in _eff.get('folds', [])]
    MODELOS.append({
        'nombre': 'EfficientNet-B4', 'tipo': 'Deep Learning', 'nivel': 'patch',
        'f1_mean': ag.get('mean_f1_thr05', np.nan),
        'f1_std':  ag.get('std_f1_thr05',  np.nan),
        'auc_mean': ag.get('mean_auc_roc', np.nan),
        'aupr_mean': ag.get('mean_auc_pr', np.nan),
        'iou_mean': np.nan,
        'folds_f1': [f['f1_thr05'] for f in _eff.get('folds', [])],
    })

if _unet_folds:
    folds_f1 = [r['f1_pixel_thr05'] for r in _unet_folds]
    MODELOS.append({
        'nombre': 'U-Net ResNet-34', 'tipo': 'Deep Learning', 'nivel': 'píxel',
        'f1_mean': float(np.mean(folds_f1)),
        'f1_std':  float(np.std(folds_f1)),
        'auc_mean': float(np.mean([r['auc_roc'] for r in _unet_folds])),
        'aupr_mean': float(np.mean([r['auc_pr']  for r in _unet_folds])),
        'iou_mean': float(np.mean([r['iou_thr05'] for r in _unet_folds])),
        'dice_mean': float(np.mean([r['dice_thr05'] for r in _unet_folds])),
        'folds_f1': folds_f1,
    })

MODELOS.sort(key=lambda x: x['f1_mean'], reverse=True)
print(f'✅ {len(MODELOS)} modelos cargados')


## 2. Tabla comparativa completa

In [ ]:
# ── Celda 2: Tabla de métricas ────────────────────────────────────────────────
print('='*80)
print(f'  {"Modelo":<22} {"Tipo":<15} {"F1":>7} {"±std":>6} {"AUC-ROC":>8} {"AUC-PR":>7} {"IoU":>6}  Nivel')
print('='*80)
for m in MODELOS:
    iou_s  = f'{m["iou_mean"]:.4f}'  if not np.isnan(m.get('iou_mean', np.nan)) else '  —   '
    auc_s  = f'{m["auc_mean"]:.4f}'  if not np.isnan(m.get('auc_mean', np.nan)) else '  —   '
    aupr_s = f'{m["aupr_mean"]:.4f}' if not np.isnan(m.get('aupr_mean', np.nan)) else '  —   '
    print(f'  {m["nombre"]:<22} {m["tipo"]:<15} {m["f1_mean"]:>7.4f} {m["f1_std"]:>6.4f} '
          f'{auc_s:>8} {aupr_s:>7} {iou_s:>6}  {m["nivel"]}')
print('='*80)
print()

# U-Net adicional: Dice
for m in MODELOS:
    if m['nombre'] == 'U-Net ResNet-34' and 'dice_mean' in m:
        print(f'  U-Net — Dice@0.5: {m["dice_mean"]:.4f}  (métrica pixel-level adicional)')


## 3. Gráficas de barras con barras de error

In [ ]:
# ── Celda 3: Panel de barras F1 + AUC ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

nombres = [m['nombre']   for m in MODELOS]
f1s     = [m['f1_mean']  for m in MODELOS]
stds    = [m['f1_std']   for m in MODELOS]
aucs    = [m['auc_mean'] for m in MODELOS]
cols    = [COLORES.get(m['nombre'], '#94A3B8') for m in MODELOS]

# Panel F1
ax = axes[0]
bars = ax.barh(range(len(nombres)), f1s, xerr=stds, color=cols,
               edgecolor='white', capsize=5, height=0.6)
ax.set_yticks(range(len(nombres))); ax.set_yticklabels(nombres, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('F1-Score (5-Fold CV, media ± std)', fontsize=11)
ax.set_title('F1-Score por modelo', fontsize=12, fontweight='bold')
for i, (v, s) in enumerate(zip(f1s, stds)):
    ax.text(v + s + 0.005, i, f'{v:.4f}', va='center', fontsize=9)
ax.set_xlim(0, 1.15)
ax.grid(axis='x', linestyle='--', alpha=0.3)

# Panel AUC-ROC
ax = axes[1]
aucs_valid  = [(i, v) for i, v in enumerate(aucs) if not np.isnan(v)]
idxs  = [i for i, _ in aucs_valid]
vals  = [v for _, v in aucs_valid]
cols2 = [cols[i] for i in idxs]
noms2 = [nombres[i] for i in idxs]
ax.barh(range(len(noms2)), vals, color=cols2, edgecolor='white', height=0.6)
ax.set_yticks(range(len(noms2))); ax.set_yticklabels(noms2, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('AUC-ROC (5-Fold CV)', fontsize=11)
ax.set_title('AUC-ROC por modelo', fontsize=12, fontweight='bold')
for i, v in enumerate(vals):
    ax.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=9)
ax.set_xlim(0.4, 1.08)
ax.axvline(0.5, color='gray', lw=1, ls='--', label='Azar')
ax.legend(fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.3)

# Leyenda tipos
patches = [mpatches.Patch(color='#059669', label='Clásico'),
           mpatches.Patch(color='#818CF8', label='Deep Learning (patch)'),
           mpatches.Patch(color='#F59E0B', label='Deep Learning (píxel)')]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig(OUT_DIR / 'barras_f1_auc.png', dpi=150, bbox_inches='tight')
plt.show(); print('Guardado: barras_f1_auc.png')


## 4. Box plots por fold — variabilidad entre folds

In [ ]:
# ── Celda 4: Box plots F1 por fold ───────────────────────────────────────────
modelos_con_folds = [m for m in MODELOS if len(m['folds_f1']) >= 2]

fig, ax = plt.subplots(figsize=(12, 5))
data   = [m['folds_f1']  for m in modelos_con_folds]
noms   = [m['nombre']    for m in modelos_con_folds]
cols_b = [COLORES.get(m['nombre'], '#94A3B8') for m in modelos_con_folds]

bp = ax.boxplot(data, patch_artist=True, notch=False, widths=0.5,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], cols_b):
    patch.set_facecolor(color); patch.set_alpha(0.8)

# Scatter de puntos individuales (folds)
for i, (d, c) in enumerate(zip(data, cols_b), 1):
    jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(d))
    ax.scatter([i + j for j in jitter], d, color=c, s=50, zorder=5,
               edgecolor='black', linewidth=0.5)

ax.set_xticks(range(1, len(noms)+1))
ax.set_xticklabels(noms, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('F1-Score por fold', fontsize=11)
ax.set_title('Distribución del F1-Score entre folds (5-Fold CV)\n'
             'Caja = IQR | Línea = mediana | Puntos = folds individuales',
             fontsize=11, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'boxplot_folds.png', dpi=150, bbox_inches='tight')
plt.show(); print('Guardado: boxplot_folds.png')


## 5. Comparación con literatura

In [ ]:
# ── Celda 5: Contexto de literatura ──────────────────────────────────────────
# Referencias publicadas sobre Landslide4Sense
LITERATURA = [
    {'ref': 'Song et al. 2025',       'f1': 0.780, 'metrica': 'F1 píxel', 'notas': 'Transformer + U-Net'},
    {'ref': 'Wang et al. 2024',       'f1': 0.740, 'metrica': 'F1 píxel', 'notas': 'Encoder dual SAR+Óptico'},
    {'ref': 'Ghorbanzadeh 2022',      'f1': 0.720, 'metrica': 'F1 píxel', 'notas': 'U-Net baseline L4S'},
    {'ref': 'Youssef et al. 2021',    'f1': 0.870, 'metrica': 'F1 patch', 'notas': 'RF + features L4S'},
]

fig, ax = plt.subplots(figsize=(13, 6))
y_pos = 0
yticks, ylabels = [], []

# Modelos del proyecto
for m in MODELOS:
    color = COLORES.get(m['nombre'], '#94A3B8')
    ax.barh(y_pos, m['f1_mean'], xerr=m['f1_std'], color=color,
            edgecolor='white', capsize=4, height=0.5, alpha=0.9)
    ax.text(m['f1_mean'] + m['f1_std'] + 0.005, y_pos,
            f'{m["f1_mean"]:.4f}', va='center', fontsize=8.5)
    yticks.append(y_pos); ylabels.append(f'{m["nombre"]} [este proyecto]')
    y_pos += 1

y_pos += 0.5  # separador

# Literatura
for lit in LITERATURA:
    ax.barh(y_pos, lit['f1'], color='#CBD5E1', edgecolor='#94A3B8',
            height=0.5, alpha=0.8, hatch='//')
    ax.text(lit['f1'] + 0.005, y_pos,
            f'{lit["f1"]:.3f} ({lit["metrica"]})', va='center', fontsize=8)
    yticks.append(y_pos); ylabels.append(f'{lit["ref"]}')
    y_pos += 1

ax.set_yticks(yticks); ax.set_yticklabels(ylabels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('F1-Score', fontsize=11)
ax.set_title('Comparación con literatura — Landslide4Sense\n'
             '(barras sólidas = este proyecto | rayadas = literatura)',
             fontsize=11, fontweight='bold')
ax.set_xlim(0, 1.1)
ax.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'comparacion_literatura.png', dpi=150, bbox_inches='tight')
plt.show(); print('Guardado: comparacion_literatura.png')


## 6. Análisis estadístico — Friedman + Wilcoxon

Evalúa si las diferencias entre modelos son estadísticamente significativas  
o podrían ser variabilidad aleatoria del fold.

- **Test de Friedman**: ANOVA no paramétrico para muestras repetidas (n_folds=5)
- **Test de Wilcoxon signed-rank**: comparación pareada entre modelos


In [ ]:
# ── Celda 6: Tests estadísticos ────────────────────────────────────────────
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

# Solo modelos con exactamente 5 folds
m5 = [m for m in MODELOS if len(m['folds_f1']) == 5]

if len(m5) < 2:
    print('⚠️  Se necesitan ≥2 modelos con 5 folds para los tests.')
else:
    # ── Friedman ──────────────────────────────────────────────────────────
    grupos = [m['folds_f1'] for m in m5]
    stat_f, p_f = friedmanchisquare(*grupos)
    print('='*60)
    print(f'  TEST DE FRIEDMAN ({len(m5)} modelos x 5 folds)')
    print('='*60)
    print(f'  χ² = {stat_f:.4f}  |  p = {p_f:.4f}')
    if p_f < 0.05:
        print('  → Diferencias SIGNIFICATIVAS entre modelos (p < 0.05) ✅')
    else:
        print('  → No se detectan diferencias significativas (p ≥ 0.05) ⚠️')

    # ── Wilcoxon pareado ──────────────────────────────────────────────────
    print()
    print('  TEST DE WILCOXON (pareado por fold):')
    print(f'  {"Par":<40} {"stat":>8} {"p-valor":>9} {"Sig."}')
    print('  ' + '-'*65)
    resultados_w = []
    for m_a, m_b in combinations(m5, 2):
        try:
            s, p = wilcoxon(m_a['folds_f1'], m_b['folds_f1'])
        except Exception:
            s, p = np.nan, 1.0
        sig = '✅ *' if p < 0.05 else ''
        par = f'{m_a["nombre"]} vs {m_b["nombre"]}'
        print(f'  {par:<40} {s:>8.3f} {p:>9.4f}  {sig}')
        resultados_w.append({'par': par, 'stat': s, 'p': p})

    # ── Heatmap de p-valores ──────────────────────────────────────────────
    n = len(m5)
    pmat = np.ones((n, n))
    nom5 = [m['nombre'] for m in m5]
    for i, j in combinations(range(n), 2):
        r = [r for r in resultados_w if m5[i]['nombre'] in r['par'] and m5[j]['nombre'] in r['par']]
        if r:
            pmat[i, j] = r[0]['p']
            pmat[j, i] = r[0]['p']

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(pmat, cmap='RdYlGn_r', vmin=0, vmax=0.1)
    ax.set_xticks(range(n)); ax.set_xticklabels(nom5, rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(n)); ax.set_yticklabels(nom5, fontsize=9)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{pmat[i,j]:.3f}' if i != j else '—',
                    ha='center', va='center', fontsize=8,
                    color='white' if pmat[i,j] < 0.05 else 'black')
    plt.colorbar(im, ax=ax, label='p-valor Wilcoxon')
    ax.set_title('Significancia estadística entre modelos\n'
                 '(verde = p < 0.05 = diferencia significativa)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'wilcoxon_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show(); print('Guardado: wilcoxon_heatmap.png')


## 7. Resumen final

In [ ]:
# ── Celda 7: Resumen ─────────────────────────────────────────────────────────
mejor = max(MODELOS, key=lambda x: x['f1_mean'])
print('='*60)
print('  RESUMEN — COMPARACIÓN DE MODELOS L4S 5-FOLD')
print('='*60)
print(f'\n  Mejor modelo: {mejor["nombre"]}')
print(f'    F1 = {mejor["f1_mean"]:.4f} ± {mejor["f1_std"]:.4f}')
if not np.isnan(mejor.get('auc_mean', np.nan)):
    print(f'    AUC-ROC = {mejor["auc_mean"]:.4f}')
print(f'    Nivel de evaluación: {mejor["nivel"]}')
print()
print('  Ranking completo por F1:')
for i, m in enumerate(MODELOS, 1):
    print(f'    {i}. {m["nombre"]:<22} F1={m["f1_mean"]:.4f} ± {m["f1_std"]:.4f}  ({m["tipo"]})')
print()
print(f'  Archivos guardados en: {OUT_DIR}')
print('    - barras_f1_auc.png')
print('    - boxplot_folds.png')
print('    - comparacion_literatura.png')
print('    - wilcoxon_heatmap.png')
print('='*60)
